# 🧹 Notebook 1 - Limpeza de Dados (Bronze → Silver)

## 1. Introdução e Objetivos

Este notebook implementa o pipeline de **limpeza e padronização** dos dados brutos (camada **Bronze**) para a camada **Silver**.

### Objetivo
Transformar os arquivos CSV crus em datasets confiáveis e padronizados no formato **Parquet**, prontos para as transformações analíticas da camada Gold.

### 1.1 Fonte dos Dados

O dataset utilizado é o **Online Shop 2024**, um conjunto de dados sintético que simula as operações de uma loja virtual durante o ano de 2024. Os dados foram obtidos publicamente e representam transações de e-commerce incluindo clientes, pedidos, produtos, pagamentos, entregas, avaliações e fornecedores.

### 1.2 Modelo Relacional e Composição das Camadas

O dataset é **relacional**, composto por **8 tabelas independentes** que se relacionam por meio de chaves estrangeiras. Abaixo está a descrição de cada tabela, sua chave primária (PK), chaves estrangeiras (FK) e seu papel no modelo:

| Tabela | Chave Primária | Chaves Estrangeiras | Descrição |
|---|---|---|---|
| `customers.csv` | `customer_id` |  | Cadastro de clientes da loja |
| `orders.csv` | `order_id` | `customer_id → customers` | Pedidos realizados pelos clientes |
| `order_items.csv` | `order_item_id` | `order_id → orders`, `product_id → products` | Itens individuais de cada pedido |
| `products.csv` | `product_id` | `supplier_id → suppliers` | Catálogo de produtos disponíveis |
| `payment.csv` | `payment_id` | `order_id → orders` | Transações de pagamento dos pedidos |
| `shipments.csv` | `shipment_id` | `order_id → orders` | Dados de envio e entrega dos pedidos |
| `reviews.csv` | `review_id` | `customer_id → customers`, `product_id → products` | Avaliações de produtos feitas por clientes |
| `suppliers.csv` | `supplier_id` |  | Fornecedores dos produtos do catálogo |

**Relacionamentos principais:**
- Um **cliente** pode realizar múltiplos **pedidos** (`customers` 1→N `orders`)
- Um **pedido** contém múltiplos **itens** (`orders` 1→N `order_items`)
- Cada **pedido** tem um **pagamento** e um **envio** associado (`orders` 1→1 `payment`, `orders` 1→1 `shipments`)
- Um **produto** pertence a um **fornecedor** e pode aparecer em múltiplas **avaliações** (`products` N→1 `suppliers`, `products` 1→N `reviews`)

**Estrutura das camadas:**
- **Bronze** (`dataset_bronze/`): 8 arquivos `.csv` originais, preservados sem modificações, representando os dados como chegaram da fonte.
- **Silver** (`dataset_silver/`): 8 arquivos `.parquet` correspondentes às tabelas bronze, após diagnóstico, limpeza e padronização de tipos. A estrutura de tabelas é mantida - não há fusão ou agregação nesta camada.
- **Gold** (`dataset_gold/`): 1 arquivo `.parquet` com base analítica orientada a clientes e pedidos, gerada no Notebook 2 a partir das tabelas silver.

### 1.3 Escopo do Notebook de Limpeza

Para cada tabela do dataset, o processo segue 4 etapas:
1. **Carregamento** dos dados brutos da pasta `dataset_bronze/`
2. **Diagnóstico de qualidade** - análise de tipos, nulos, duplicatas, cardinalidade e anomalias
3. **Limpeza e transformação** - aplicação das correções definidas no plano de limpeza
4. **Exportação** - salvamento em formato `.parquet` na pasta `dataset_silver/`

obs: Para transformar em Parquet talvez seja necessário instalar: `pip install pyarrow`

---
## 2. Imports e Configurações

In [1]:
# === Imports ===
import pandas as pd
import os
from pathlib import Path

# === Configurações de exibição ===
pd.set_option('display.max_columns', None)   # Exibir todas as colunas
pd.set_option('display.max_rows', 20)         # Limitar linhas exibidas
pd.set_option('display.width', 200)            # Largura da saída
pd.set_option('display.float_format', '{:.2f}'.format)  # 2 casas decimais

# === Caminhos ===
PATH_BRONZE = Path('../datasets/dataset_bronze/')
PATH_SILVER = Path('../datasets/dataset_silver/')

# Garantir que a pasta Silver existe
PATH_SILVER.mkdir(parents=True, exist_ok=True)

print('✅ Imports e configurações carregados com sucesso.')
print(f'📂 Bronze: {PATH_BRONZE.resolve()}')
print(f'📂 Silver: {PATH_SILVER.resolve()}')

✅ Imports e configurações carregados com sucesso.
📂 Bronze: C:\Users\Nogueira\Documents\GitHub\Trabalho_ciencia_de_dados\datasets\dataset_bronze
📂 Silver: C:\Users\Nogueira\Documents\GitHub\Trabalho_ciencia_de_dados\datasets\dataset_silver


---
## 3. `customers.csv` - Diagnóstico e Limpeza

**Tabela:** Cadastro de clientes  
**Colunas esperadas:** `customer_id`, `first_name`, `last_name`, `address`, `email`, `phone_number`  
**Registros esperados:** 10.000

### 3.1 Carregamento

In [2]:
# Carregamento da tabela customers
df_customers = pd.read_csv(PATH_BRONZE / 'customers.csv')
print(f'Shape: {df_customers.shape}')
df_customers.head()

Shape: (10000, 6)


,customer_id,first_name,last_name,address,email,phone_number
0,1,James,Smith,"123 Main St, Springfield, IL",jsmith1@customer.com,555-190-3233
1,2,James,Johnson,"123 Main St, Springfield, IL",jjohnson2@customer.com,555-525-8357
2,3,James,Williams,"123 Main St, Springfield, IL",jwilliams3@customer.com,555-933-4447
3,4,James,Brown,"123 Main St, Springfield, IL",jbrown4@customer.com,555-897-0326
4,5,James,Jones,"123 Main St, Springfield, IL",jjones5@customer.com,555-762-5836


### 3.2 Diagnóstico de Qualidade

In [3]:
# --- Tipos de dados ---
print('=== TIPOS DE DADOS ===')
print(df_customers.dtypes)
print()

=== TIPOS DE DADOS ===
customer_id     int64
first_name        str
last_name         str
address           str
email             str
phone_number      str
dtype: object



In [4]:
# --- Valores nulos ---
print('=== VALORES NULOS ===')
print(df_customers.isnull().sum())
print()

=== VALORES NULOS ===
customer_id     0
first_name      0
last_name       0
address         0
email           0
phone_number    0
dtype: int64



In [5]:
# --- Duplicatas (linhas completas) ---
n_duplicatas = df_customers.duplicated().sum()
print(f'=== DUPLICATAS (full row) === {n_duplicatas}')
print()

=== DUPLICATAS (full row) === 0



In [6]:
# --- Cardinalidade (valores únicos por coluna) ---
print('=== CARDINALIDADE ===')
for col in df_customers.columns:
    print(f'  {col}: {df_customers[col].nunique()} únicos')
print()

=== CARDINALIDADE ===
  customer_id: 10000 únicos
  first_name: 3 únicos
  last_name: 70 únicos
  address: 59 únicos
  email: 10000 únicos
  phone_number: 9995 únicos



In [7]:
# --- Análise detalhada: first_name (baixa cardinalidade) ---
print('=== DISTRIBUIÇÃO first_name ===')
print(df_customers['first_name'].value_counts())
print()

=== DISTRIBUIÇÃO first_name ===
first_name
James    4130
Mary     4130
John     1740
Name: count, dtype: int64



In [8]:
# --- Análise detalhada: phone_number duplicados ---
print('=== PHONE_NUMBER DUPLICADOS ===')
phone_dupes = df_customers[df_customers.duplicated(subset=['phone_number'], keep=False)]
print(f'Total de registros com phone_number repetido: {len(phone_dupes)}')
print(phone_dupes.sort_values('phone_number')[['customer_id', 'email', 'phone_number']])

=== PHONE_NUMBER DUPLICADOS ===
Total de registros com phone_number repetido: 10
      customer_id                        email  phone_number
6977         6978      malice6978@customer.com  555-085-3339
9205         9206  jchristina9206@customer.com  555-085-3339
5576         5577   mlawrence5577@customer.com  555-093-7369
5799         5800    mabigail5800@customer.com  555-093-7369
2027         2028        jamy2028@customer.com  555-259-8241
7581         7582   mvictoria7582@customer.com  555-259-8241
3410         3411     jphilip3411@customer.com  555-345-8126
4011         4012   jvictoria4012@customer.com  555-345-8126
6699         6700       mruth6700@customer.com  555-818-1841
9041         9042      jlopez9042@customer.com  555-818-1841


### 3.3 Justificativas e Decisões de Limpeza

| # | Observação | Decisão | Justificativa |
|---|---|---|---|
| 1 | **Nenhum valor nulo** encontrado (0 em todas as colunas) | ✅ Nenhuma ação | Dataset completo |
| 2 | **Nenhuma duplicata** de linha completa | ✅ Nenhuma ação | IDs únicos |
| 3 | **Baixa cardinalidade em `first_name`** - apenas 3 valores únicos (James, John, Mary) para 10.000 clientes | 📝 Documentar como limitação | Característica do dataset sintético; não é um erro de dados |
| 4 | **Baixa cardinalidade em `address`** - apenas 59 endereços únicos para 10.000 clientes | 📝 Documentar como limitação | Característica do dataset sintético |
| 5 | **5 phone_numbers duplicados** - 10 registros compartilham 5 números (cada phone aparece 2×) | 📝 Manter e documentar | Os `customer_id` e `email` são distintos - podem ser clientes que compartilham telefone (ex: mesmo domicílio) |
| 6 | **Formato do phone_number** consistente (`555-XXX-XXXX`) em 100% dos registros | ✅ Nenhuma ação | Padronizado |
| 7 | **Tipos de dados** adequados (`int64` para ID, `object` para texto) | ✅ Nenhuma ação | Corretos |
| 8 | **Colunas já em `snake_case`** | ✅ Nenhuma ação | Padrão seguido |

> **Conclusão:** O dataset `customers.csv` está **limpo**. Nenhuma transformação estrutural é necessária. As limitações do dataset sintético (baixa variabilidade de nomes e endereços) e os telefones duplicados são documentados como observações.

### 3.4 Limpeza e Validação

In [ ]:
# customers.csv não requer transformações - dataset já está limpo.
# Apenas validação final antes da exportação.

print('=== VALIDAÇÃO FINAL - customers ===')
print(f'Shape: {df_customers.shape}')
print(f'Nulos totais: {df_customers.isnull().sum().sum()}')
print(f'Duplicatas: {df_customers.duplicated().sum()}')
print(f'customer_id únicos: {df_customers["customer_id"].nunique()}')
print()
print(df_customers.dtypes)
print()
print('✅ customers - pronto para exportação Silver.')

=== VALIDAÇÃO FINAL — customers ===
Shape: (10000, 6)
Nulos totais: 0
Duplicatas: 0
customer_id únicos: 10000

customer_id     int64
first_name        str
last_name         str
address           str
email             str
phone_number      str
dtype: object

✅ customers — pronto para exportação Silver.


---
## 4. `order_items.csv` - Diagnóstico e Limpeza

**Tabela:** Itens de cada pedido  
**Colunas esperadas:** `order_item_id`, `order_id`, `product_id`, `quantity`, `price_at_purchase`  
**Registros esperados:** 20.000

### 4.1 Carregamento

In [10]:
# Carregamento da tabela order_items
df_order_items = pd.read_csv(PATH_BRONZE / 'order_items.csv')
print(f'Shape: {df_order_items.shape}')
df_order_items.head()

Shape: (20000, 5)


,order_item_id,order_id,product_id,quantity,price_at_purchase
0,1,6550,1032,1,342.92
1,2,7324,1695,1,955.86
2,3,11952,1962,1,909.45
3,4,8938,1958,1,984.91
4,5,4238,1880,1,649.18


### 4.2 Diagnóstico de Qualidade

In [11]:
# --- Tipos de dados ---
print('=== TIPOS DE DADOS ===')
print(df_order_items.dtypes)
print()

=== TIPOS DE DADOS ===
order_item_id          int64
order_id               int64
product_id             int64
quantity               int64
price_at_purchase    float64
dtype: object



In [12]:
# --- Valores nulos ---
print('=== VALORES NULOS ===')
print(df_order_items.isnull().sum())
print()

=== VALORES NULOS ===
order_item_id        0
order_id             0
product_id           0
quantity             0
price_at_purchase    0
dtype: int64



In [13]:
# --- Duplicatas (linhas completas) ---
n_duplicatas = df_order_items.duplicated().sum()
print(f'=== DUPLICATAS (full row) === {n_duplicatas}')
print()

=== DUPLICATAS (full row) === 0



In [14]:
# --- Cardinalidade ---
print('=== CARDINALIDADE ===')
for col in df_order_items.columns:
    print(f'  {col}: {df_order_items[col].nunique()} únicos')
print()

=== CARDINALIDADE ===
  order_item_id: 20000 únicos
  order_id: 15000 únicos
  product_id: 1997 únicos
  quantity: 10 únicos
  price_at_purchase: 16399 únicos



In [15]:
# --- Estatísticas descritivas ---
print('=== ESTATÍSTICAS DESCRITIVAS ===')
df_order_items.describe()

=== ESTATÍSTICAS DESCRITIVAS ===


,order_item_id,order_id,product_id,quantity,price_at_purchase
count,20000.00,20000.00,20000.00,20000.00,20000.00
mean,10000.50,7500.04,1000.14,4.69,387.01
std,5773.65,4327.06,573.55,2.92,324.77
min,1.00,1.00,1.00,1.00,0.00
25%,5000.75,3760.75,508.00,2.00,32.11
50%,10000.50,7486.50,996.50,4.00,347.20
75%,15000.25,11250.25,1490.00,7.00,675.14
max,20000.00,15000.00,2000.00,10.00,999.82


In [16]:
# --- Análise: registros com price_at_purchase == 0 ---
print('=== REGISTROS COM PREÇO ZERO ===')
zero_price = df_order_items[df_order_items['price_at_purchase'] == 0.0]
print(f'Total de registros com price_at_purchase = 0: {len(zero_price)}')
print(zero_price)

=== REGISTROS COM PREÇO ZERO ===
Total de registros com price_at_purchase = 0: 2
       order_item_id  order_id  product_id  quantity  price_at_purchase
15000          15001      2800        1851         1               0.00
15001          15002      4799          32         1               0.00


In [17]:
# --- Análise: range de quantity ---
print('=== RANGE DE QUANTITY ===')
print(f'Mínimo: {df_order_items["quantity"].min()}')
print(f'Máximo: {df_order_items["quantity"].max()}')
print(f'Valores <= 0: {(df_order_items["quantity"] <= 0).sum()}')

=== RANGE DE QUANTITY ===
Mínimo: 1
Máximo: 10
Valores <= 0: 0


### 4.3 Justificativas e Decisões de Limpeza

| # | Observação | Decisão | Justificativa |
|---|---|---|---|
| 1 | **Nenhum valor nulo** encontrado (0 em todas as colunas) | ✅ Nenhuma ação | Dataset completo |
| 2 | **Nenhuma duplicata** de linha completa | ✅ Nenhuma ação | IDs únicos |
| 3 | **2 registros com `price_at_purchase` = 0.0** (order_item_id 15001 e 15002) - itens com quantidade 1 e preço zero | 📝 Manter e documentar | Podem representar itens promocionais, brindes ou cortesias. Remover esses registros poderia comprometer a integridade referencial com a tabela de pedidos. A anomalia é registrada para análise futura. |
| 4 | **`quantity`** no range de 1 a 10, sem valores negativos ou zero | ✅ Nenhuma ação | Valores válidos |
| 5 | **Tipos de dados** adequados (`int64` para IDs/qty, `float64` para preço) | ✅ Nenhuma ação | Corretos |
| 6 | **Colunas já em `snake_case`** | ✅ Nenhuma ação | Padrão seguido |

> **Conclusão:** O dataset `order_items.csv` está **limpo**. Os 2 registros com preço zero são mantidos e documentados como anomalia - a decisão de mantê-los preserva a integridade referencial e permite análise posterior na camada Gold.

### 4.4 Limpeza e Validação

In [ ]:
# order_items.csv não requer transformações estruturais.
# Os 2 registros com price_at_purchase == 0 são mantidos (documentados acima).
# Apenas validação final antes da exportação.

print('=== VALIDAÇÃO FINAL - order_items ===')
print(f'Shape: {df_order_items.shape}')
print(f'Nulos totais: {df_order_items.isnull().sum().sum()}')
print(f'Duplicatas: {df_order_items.duplicated().sum()}')
print(f'order_item_id únicos: {df_order_items["order_item_id"].nunique()}')
print(f'Registros com preço zero: {(df_order_items["price_at_purchase"] == 0).sum()} (documentados)')
print()
print(df_order_items.dtypes)
print()
print('✅ order_items - pronto para exportação Silver.')

=== VALIDAÇÃO FINAL — order_items ===
Shape: (20000, 5)
Nulos totais: 0
Duplicatas: 0
order_item_id únicos: 20000
Registros com preço zero: 2 (documentados)

order_item_id          int64
order_id               int64
product_id             int64
quantity               int64
price_at_purchase    float64
dtype: object

✅ order_items — pronto para exportação Silver.


---
## 5. `orders.csv` - Diagnóstico e Limpeza

**Tabela:** Pedidos realizados  
**Colunas esperadas:** `order_id`, `order_date`, `customer_id`, `total_price`  
**Registros esperados:** 15.000

### 5.1 Carregamento

In [19]:
# Carregamento da tabela orders
df_orders = pd.read_csv(PATH_BRONZE / 'orders.csv')
print(f'Shape: {df_orders.shape}')
df_orders.head()

Shape: (15000, 4)


,order_id,order_date,customer_id,total_price
0,7324,2023-11-27,2160,955.86
1,8938,2024-08-31,8497,984.91
2,4238,2023-12-05,6295,649.18
3,10944,2024-02-22,2936,54.83
4,11075,2024-08-14,2994,320.24


### 5.2 Diagnóstico de Qualidade

In [20]:
# --- Tipos de dados ---
print('=== TIPOS DE DADOS ===')
print(df_orders.dtypes)
print()

=== TIPOS DE DADOS ===
order_id         int64
order_date         str
customer_id      int64
total_price    float64
dtype: object



In [21]:
# --- Valores nulos ---
print('=== VALORES NULOS ===')
print(df_orders.isnull().sum())
print()

=== VALORES NULOS ===
order_id       0
order_date     0
customer_id    0
total_price    0
dtype: int64



In [22]:
# --- Duplicatas (linhas completas) ---
n_duplicatas = df_orders.duplicated().sum()
print(f'=== DUPLICATAS (full row) === {n_duplicatas}')
print()

=== DUPLICATAS (full row) === 0



In [23]:
# --- Cardinalidade ---
print('=== CARDINALIDADE ===')
for col in df_orders.columns:
    print(f'  {col}: {df_orders[col].nunique()} únicos')
print()

=== CARDINALIDADE ===
  order_id: 15000 únicos
  order_date: 366 únicos
  customer_id: 10000 únicos
  total_price: 14741 únicos



In [24]:
# --- Tipo atual de order_date ---
print('=== ANÁLISE: order_date ===')
print(f'Tipo atual: {df_orders["order_date"].dtype}')
print(f'Exemplo de valores: {df_orders["order_date"].head(3).tolist()}')
print(f'Mínimo: {df_orders["order_date"].min()}')
print(f'Máximo: {df_orders["order_date"].max()}')

=== ANÁLISE: order_date ===
Tipo atual: str
Exemplo de valores: ['2023-11-27', '2024-08-31', '2023-12-05']
Mínimo: 2023-11-05
Máximo: 2024-11-04


In [25]:
# --- Estatísticas de total_price ---
print('=== ANÁLISE: total_price ===')
print(f'Mínimo: {df_orders["total_price"].min()}')
print(f'Máximo: {df_orders["total_price"].max()}')
print(f'Valores <= 0: {(df_orders["total_price"] <= 0).sum()}')
print()
df_orders['total_price'].describe()

=== ANÁLISE: total_price ===
Mínimo: 20.46
Máximo: 10082.72
Valores <= 0: 0



count   15000.00
mean     2833.05
std      2275.53
min        20.46
25%       932.52
50%      2207.07
75%      4249.19
max     10082.72
Name: total_price, dtype: float64

### 5.3 Justificativas e Decisões de Limpeza

| # | Observação | Decisão | Justificativa |
|---|---|---|---|
| 1 | **Nenhum valor nulo** encontrado (0 em todas as colunas) | ✅ Nenhuma ação | Dataset completo |
| 2 | **Nenhuma duplicata** de linha completa | ✅ Nenhuma ação | IDs únicos |
| 3 | **`order_date` está como `object` (string)** - formato `YYYY-MM-DD` consistente | 🔧 Converter para `datetime64` | Permite operações temporais, filtros por período e joins com outras tabelas de datas. Tipo `object` impede cálculos com datas. |
| 4 | **Range de datas** de 2023-11-05 a 2024-11-04 (≈ 1 ano) | ✅ Nenhuma ação | Período consistente e coerente |
| 5 | **`total_price`** sem valores ≤ 0 | ✅ Nenhuma ação | Valores válidos |
| 6 | **Colunas já em `snake_case`** | ✅ Nenhuma ação | Padrão seguido |

> **Conclusão:** A única transformação necessária é a **conversão de `order_date`** de `object` para `datetime64`. Esta conversão é essencial para viabilizar análises temporais na camada Gold (tendências, sazonalidade, cohorts).

### 5.4 Limpeza e Validação

In [26]:
# === Limpeza: Conversão de order_date para datetime64 ===
print('=== ANTES DA CONVERSÃO ===')
print(f'order_date dtype: {df_orders["order_date"].dtype}')
print()

df_orders['order_date'] = pd.to_datetime(df_orders['order_date'])

print('=== APÓS A CONVERSÃO ===')
print(f'order_date dtype: {df_orders["order_date"].dtype}')
print(f'Mínimo: {df_orders["order_date"].min()}')
print(f'Máximo: {df_orders["order_date"].max()}')

=== ANTES DA CONVERSÃO ===
order_date dtype: str

=== APÓS A CONVERSÃO ===
order_date dtype: datetime64[us]
Mínimo: 2023-11-05 00:00:00
Máximo: 2024-11-04 00:00:00


In [ ]:
# === Validação final ===
print('=== VALIDAÇÃO FINAL - orders ===')
print(f'Shape: {df_orders.shape}')
print(f'Nulos totais: {df_orders.isnull().sum().sum()}')
print(f'Duplicatas: {df_orders.duplicated().sum()}')
print(f'order_id únicos: {df_orders["order_id"].nunique()}')
print()
print(df_orders.dtypes)
print()
print('✅ orders - pronto para exportação Silver.')

=== VALIDAÇÃO FINAL — orders ===
Shape: (15000, 4)
Nulos totais: 0
Duplicatas: 0
order_id únicos: 15000

order_id                int64
order_date     datetime64[us]
customer_id             int64
total_price           float64
dtype: object

✅ orders — pronto para exportação Silver.


---
## 6. `payment.csv` - Diagnóstico e Limpeza

**Tabela:** Pagamentos dos pedidos  
**Colunas esperadas:** `payment_id`, `order_id`, `payment_method`, `amount`, `transaction_status`  
**Registros esperados:** 15.000

> **Nota:** O README do projeto menciona uma coluna `payment_date`, porém ela **não está presente** no arquivo CSV.

### 6.1 Carregamento

In [28]:
# Carregamento da tabela payment
df_payment = pd.read_csv(PATH_BRONZE / 'payment.csv')
print(f'Shape: {df_payment.shape}')
df_payment.head()

Shape: (15000, 5)


,payment_id,order_id,payment_method,amount,transaction_status
0,1,1,Credit Card,232.15,Completed
1,2,2,Credit Card,454.44,Completed
2,3,3,Credit Card,26.14,Completed
3,4,4,Credit Card,985.70,Completed
4,5,5,Credit Card,733.32,Completed


### 6.2 Diagnóstico de Qualidade

In [29]:
# --- Tipos de dados ---
print('=== TIPOS DE DADOS ===')
print(df_payment.dtypes)
print()

=== TIPOS DE DADOS ===
payment_id              int64
order_id                int64
payment_method            str
amount                float64
transaction_status        str
dtype: object



In [30]:
# --- Valores nulos ---
print('=== VALORES NULOS ===')
print(df_payment.isnull().sum())
print()

=== VALORES NULOS ===
payment_id            0
order_id              0
payment_method        0
amount                0
transaction_status    0
dtype: int64



In [31]:
# --- Duplicatas (linhas completas) ---
n_duplicatas = df_payment.duplicated().sum()
print(f'=== DUPLICATAS (full row) === {n_duplicatas}')
print()

=== DUPLICATAS (full row) === 0



In [32]:
# --- Cardinalidade ---
print('=== CARDINALIDADE ===')
for col in df_payment.columns:
    print(f'  {col}: {df_payment[col].nunique()} únicos')
print()

=== CARDINALIDADE ===
  payment_id: 15000 únicos
  order_id: 15000 únicos
  payment_method: 1 únicos
  amount: 7806 únicos
  transaction_status: 3 únicos



In [33]:
# --- Análise: payment_method ---
print('=== DISTRIBUIÇÃO payment_method ===')
print(df_payment['payment_method'].value_counts())
print()

=== DISTRIBUIÇÃO payment_method ===
payment_method
Credit Card    15000
Name: count, dtype: int64



In [34]:
# --- Análise: transaction_status ---
print('=== DISTRIBUIÇÃO transaction_status ===')
print(df_payment['transaction_status'].value_counts())
print()

=== DISTRIBUIÇÃO transaction_status ===
transaction_status
Completed    12000
Pending       1500
Failed        1500
Name: count, dtype: int64



In [35]:
# --- Análise: amount ---
print('=== ANÁLISE: amount ===')
print(f'Mínimo: {df_payment["amount"].min()}')
print(f'Máximo: {df_payment["amount"].max()}')
print(f'Valores <= 0: {(df_payment["amount"] <= 0).sum()}')
print()
df_payment['amount'].describe()

=== ANÁLISE: amount ===
Mínimo: 0.01
Máximo: 999.9
Valores <= 0: 0



count   15000.00
mean      358.98
std       312.45
min         0.01
25%        75.29
50%       248.57
75%       603.20
max       999.90
Name: amount, dtype: float64

In [36]:
# --- Verificação: coluna payment_date ausente ---
print('=== VERIFICAÇÃO: payment_date ===')
print(f'Colunas presentes no CSV: {df_payment.columns.tolist()}')
print(f'payment_date presente? {"payment_date" in df_payment.columns}')
print()
print('⚠️ Confirmado: a coluna payment_date mencionada no README NÃO existe no arquivo CSV.')

=== VERIFICAÇÃO: payment_date ===
Colunas presentes no CSV: ['payment_id', 'order_id', 'payment_method', 'amount', 'transaction_status']
payment_date presente? False

⚠️ Confirmado: a coluna payment_date mencionada no README NÃO existe no arquivo CSV.


### 6.3 Justificativas e Decisões de Limpeza

| # | Observação | Decisão | Justificativa |
|---|---|---|---|
| 1 | **Nenhum valor nulo** encontrado (0 em todas as colunas) | ✅ Nenhuma ação | Dataset completo |
| 2 | **Nenhuma duplicata** de linha completa | ✅ Nenhuma ação | IDs únicos |
| 3 | **Coluna `payment_date` AUSENTE** - o README define essa coluna, mas ela não existe no CSV | 📝 Documentar divergência | Divergência entre documentação e dados reais. Não criaremos uma coluna fictícia - documentamos a ausência como limitação do dataset. |
| 4 | **`payment_method`** com cardinalidade 1 - somente `Credit Card` | 📝 Documentar como limitação | Não é um erro, mas limita análises por método de pagamento. Característica do dataset sintético. |
| 5 | **`transaction_status`** com 3 valores (Completed: ~12k, Pending: ~1.5k, Failed: ~1.5k) | ✅ Nenhuma ação | Distribuição consistente e valores válidos |
| 6 | **`amount`** no range de 0.01 a 999.90, sem valores negativos | ✅ Nenhuma ação | Valores válidos |
| 7 | **Tipos de dados** adequados (`int64` para IDs, `float64` para amount, `object` para texto) | ✅ Nenhuma ação | Corretos |
| 8 | **Colunas já em `snake_case`** | ✅ Nenhuma ação | Padrão seguido |

> **Conclusão:** O dataset `payment.csv` está **limpo** - nenhuma transformação estrutural é necessária. As limitações (ausência de `payment_date` e cardinalidade 1 em `payment_method`) são documentadas como observações.

### 6.4 Limpeza e Validação

In [ ]:
# payment.csv não requer transformações - dataset já está limpo.
# Apenas validação final antes da exportação.

print('=== VALIDAÇÃO FINAL - payment ===')
print(f'Shape: {df_payment.shape}')
print(f'Nulos totais: {df_payment.isnull().sum().sum()}')
print(f'Duplicatas: {df_payment.duplicated().sum()}')
print(f'payment_id únicos: {df_payment["payment_id"].nunique()}')
print()
print(df_payment.dtypes)
print()
print('✅ payment - pronto para exportação Silver.')

=== VALIDAÇÃO FINAL — payment ===
Shape: (15000, 5)
Nulos totais: 0
Duplicatas: 0
payment_id únicos: 15000

payment_id              int64
order_id                int64
payment_method            str
amount                float64
transaction_status        str
dtype: object

✅ payment — pronto para exportação Silver.


---
## 7. `products.csv` - Diagnóstico e Limpeza

**Tabela:** Catálogo de produtos  
**Colunas esperadas:** `product_id`, `product_name`, `category`, `price`, `supplier_id`  
**Registros esperados:** 2.000

### 7.1 Carregamento

In [38]:
# Carregamento da tabela products
df_products = pd.read_csv(PATH_BRONZE / 'products.csv')
print(f'Shape: {df_products.shape}')
df_products.head()

Shape: (2000, 5)


,product_id,product_name,category,price,supplier_id
0,1,Office Chair,Furniture,445.01,501
1,2,Coffee Maker,Home & Kitchen,937.29,502
2,3,Document Scanner,Electronics,940.02,503
3,4,Desk Mat,Accessories,76.11,504
4,5,Tablet Stand,Accessories,388.17,505


### 7.2 Diagnóstico de Qualidade

In [39]:
# --- Tipos de dados ---
print('=== TIPOS DE DADOS ===')
print(df_products.dtypes)
print()

=== TIPOS DE DADOS ===
product_id        int64
product_name        str
category            str
price           float64
supplier_id       int64
dtype: object



In [40]:
# --- Valores nulos ---
print('=== VALORES NULOS ===')
print(df_products.isnull().sum())
print()

=== VALORES NULOS ===
product_id      0
product_name    0
category        0
price           0
supplier_id     0
dtype: int64



In [41]:
# --- Duplicatas (linhas completas) ---
n_duplicatas = df_products.duplicated().sum()
print(f'=== DUPLICATAS (full row) === {n_duplicatas}')
print()

=== DUPLICATAS (full row) === 0



In [42]:
# --- Cardinalidade ---
print('=== CARDINALIDADE ===')
for col in df_products.columns:
    print(f'  {col}: {df_products[col].nunique()} únicos')
print()

=== CARDINALIDADE ===
  product_id: 2000 únicos
  product_name: 50 únicos
  category: 4 únicos
  price: 1988 únicos
  supplier_id: 100 únicos



In [43]:
# --- Análise: product_name (baixa cardinalidade) ---
print('=== DISTRIBUIÇÃO product_name (top 10) ===')
print(df_products['product_name'].value_counts().head(10))
print()
print(f'Total nomes únicos: {df_products["product_name"].nunique()} para {len(df_products)} produtos')

=== DISTRIBUIÇÃO product_name (top 10) ===
product_name
Office Chair         40
Coffee Maker         40
Document Scanner     40
Desk Mat             40
Tablet Stand         40
Drawer Unit          40
Bath Towels          40
External SSD         40
Computer Speakers    40
Gaming Keyboard      40
Name: count, dtype: int64

Total nomes únicos: 50 para 2000 produtos


In [44]:
# --- Análise: category ---
print('=== DISTRIBUIÇÃO category ===')
print(df_products['category'].value_counts())
print()

=== DISTRIBUIÇÃO category ===
category
Electronics       720
Home & Kitchen    520
Accessories       480
Furniture         280
Name: count, dtype: int64



In [45]:
# --- Análise: price ---
print('=== ANÁLISE: price ===')
print(f'Mínimo: {df_products["price"].min()}')
print(f'Máximo: {df_products["price"].max()}')
print(f'Valores <= 0: {(df_products["price"] <= 0).sum()}')
print()
df_products['price'].describe()

=== ANÁLISE: price ===
Mínimo: 0.16
Máximo: 999.92
Valores <= 0: 0



count   2000.00
mean     504.91
std      289.00
min        0.16
25%      250.46
50%      516.58
75%      751.64
max      999.92
Name: price, dtype: float64

### 7.3 Justificativas e Decisões de Limpeza

| # | Observação | Decisão | Justificativa |
|---|---|---|---|
| 1 | **Nenhum valor nulo** encontrado | ✅ Nenhuma ação | Dataset completo |
| 2 | **Nenhuma duplicata** de linha completa | ✅ Nenhuma ação | IDs únicos |
| 3 | **Baixa cardinalidade em `product_name`** - apenas 50 nomes únicos para 2.000 produtos | 📝 Documentar como limitação | Dataset sintético com nomes repetidos mas IDs distintos |
| 4 | **`category`** com 4 valores (Accessories, Electronics, Furniture, Home & Kitchen) | ✅ Nenhuma ação | Distribuição consistente |
| 5 | **`price`** sem valores ≤ 0, range de 0.16 a 999.92 | ✅ Nenhuma ação | Valores válidos |
| 6 | **Tipos de dados** adequados e **colunas em `snake_case`** | ✅ Nenhuma ação | Corretos |

> **Conclusão:** O dataset `products.csv` está **limpo**. A baixa cardinalidade de `product_name` é documentada como limitação do dataset sintético.

### 7.4 Limpeza e Validação

In [ ]:
# products.csv não requer transformações - dataset já está limpo.
print('=== VALIDAÇÃO FINAL - products ===')
print(f'Shape: {df_products.shape}')
print(f'Nulos totais: {df_products.isnull().sum().sum()}')
print(f'Duplicatas: {df_products.duplicated().sum()}')
print(f'product_id únicos: {df_products["product_id"].nunique()}')
print()
print(df_products.dtypes)
print()
print('✅ products - pronto para exportação Silver.')

=== VALIDAÇÃO FINAL — products ===
Shape: (2000, 5)
Nulos totais: 0
Duplicatas: 0
product_id únicos: 2000

product_id        int64
product_name        str
category            str
price           float64
supplier_id       int64
dtype: object

✅ products — pronto para exportação Silver.


---
## 8. `reviews.csv` - Diagnóstico e Limpeza

**Tabela:** Avaliações de produtos  
**Colunas esperadas:** `review_id`, `product_id`, `customer_id`, `rating`, `review_text`, `review_date`  
**Registros esperados:** 1.106

### 8.1 Carregamento

In [47]:
# Carregamento da tabela reviews
df_reviews = pd.read_csv(PATH_BRONZE / 'reviews.csv')
print(f'Shape: {df_reviews.shape}')
df_reviews.head()

Shape: (1106, 6)


,review_id,product_id,customer_id,rating,review_text,review_date
0,1,16,1516,4,"Customer service was excellent, product too.",2024-03-03
1,2,1516,1516,4,"Customer service was excellent, product too.",2024-01-08
2,3,1566,9066,1,Basic functionality but reliable.,2024-03-23
3,4,66,9066,1,Basic functionality but reliable.,2024-03-25
4,5,1522,1522,3,Better than similar products I have tried.,2024-03-03


### 8.2 Diagnóstico de Qualidade

In [48]:
# --- Tipos de dados ---
print('=== TIPOS DE DADOS ===')
print(df_reviews.dtypes)
print()

=== TIPOS DE DADOS ===
review_id      int64
product_id     int64
customer_id    int64
rating         int64
review_text      str
review_date      str
dtype: object



In [49]:
# --- Valores nulos ---
print('=== VALORES NULOS ===')
print(df_reviews.isnull().sum())
print()

=== VALORES NULOS ===
review_id      0
product_id     0
customer_id    0
rating         0
review_text    0
review_date    0
dtype: int64



In [50]:
# --- Duplicatas (linhas completas) ---
n_duplicatas = df_reviews.duplicated().sum()
print(f'=== DUPLICATAS (full row) === {n_duplicatas}')
print()

=== DUPLICATAS (full row) === 0



In [51]:
# --- Cardinalidade ---
print('=== CARDINALIDADE ===')
for col in df_reviews.columns:
    print(f'  {col}: {df_reviews[col].nunique()} únicos')
print()

=== CARDINALIDADE ===
  review_id: 1106 únicos
  product_id: 158 únicos
  customer_id: 553 únicos
  rating: 5 únicos
  review_text: 79 únicos
  review_date: 348 únicos



In [52]:
# --- Análise: review_date ---
print('=== ANÁLISE: review_date ===')
print(f'Tipo atual: {df_reviews["review_date"].dtype}')
print(f'Mínimo: {df_reviews["review_date"].min()}')
print(f'Máximo: {df_reviews["review_date"].max()}')

=== ANÁLISE: review_date ===
Tipo atual: str
Mínimo: 2023-11-17
Máximo: 2024-11-16


In [53]:
# --- Análise: rating ---
print('=== DISTRIBUIÇÃO rating ===')
print(df_reviews['rating'].value_counts().sort_index())
print(f'\nRange: {df_reviews["rating"].min()} a {df_reviews["rating"].max()}')

=== DISTRIBUIÇÃO rating ===
rating
1    280
2    266
3    196
4    168
5    196
Name: count, dtype: int64

Range: 1 a 5


In [54]:
# --- Análise: review_text (baixa cardinalidade) ---
print('=== REVIEW_TEXT ===')
print(f'Total de textos únicos: {df_reviews["review_text"].nunique()} para {len(df_reviews)} reviews')
print()
print('Top 5 textos mais frequentes:')
print(df_reviews['review_text'].value_counts().head())

=== REVIEW_TEXT ===
Total de textos únicos: 79 para 1106 reviews

Top 5 textos mais frequentes:
review_text
Customer service was excellent, product too.    14
Basic functionality but reliable.               14
Better than similar products I have tried.      14
Fantastic, exactly what I needed.               14
Definitely recommend to others.                 14
Name: count, dtype: int64


### 8.3 Justificativas e Decisões de Limpeza

| # | Observação | Decisão | Justificativa |
|---|---|---|---|
| 1 | **Nenhum valor nulo** encontrado | ✅ Nenhuma ação | Dataset completo |
| 2 | **Nenhuma duplicata** de linha completa | ✅ Nenhuma ação | IDs únicos |
| 3 | **`review_date` está como `object` (string)** - formato `YYYY-MM-DD` consistente | 🔧 Converter para `datetime64` | Permite análises temporais e joins com tabelas de datas |
| 4 | **Range de datas** de 2023-11-17 a 2024-11-16 | ✅ Nenhuma ação | Consistente com o período dos pedidos |
| 5 | **`rating`** no range de 1 a 5 | ✅ Nenhuma ação | Distribuição consistente |
| 6 | **Baixa cardinalidade em `review_text`** - apenas 79 textos únicos para 1.106 reviews | 📝 Documentar como limitação | Dataset sintético com textos repetidos |
| 7 | **Colunas já em `snake_case`** | ✅ Nenhuma ação | Padrão seguido |

> **Conclusão:** A única transformação necessária é a **conversão de `review_date`** de `object` para `datetime64`. A baixa cardinalidade de `review_text` é documentada como limitação.

### 8.4 Limpeza e Validação

In [55]:
# === Limpeza: Conversão de review_date para datetime64 ===
print('=== ANTES DA CONVERSÃO ===')
print(f'review_date dtype: {df_reviews["review_date"].dtype}')
print()

df_reviews['review_date'] = pd.to_datetime(df_reviews['review_date'])

print('=== APÓS A CONVERSÃO ===')
print(f'review_date dtype: {df_reviews["review_date"].dtype}')
print(f'Mínimo: {df_reviews["review_date"].min()}')
print(f'Máximo: {df_reviews["review_date"].max()}')

=== ANTES DA CONVERSÃO ===
review_date dtype: str

=== APÓS A CONVERSÃO ===
review_date dtype: datetime64[us]
Mínimo: 2023-11-17 00:00:00
Máximo: 2024-11-16 00:00:00


In [ ]:
# === Validação final ===
print('=== VALIDAÇÃO FINAL - reviews ===')
print(f'Shape: {df_reviews.shape}')
print(f'Nulos totais: {df_reviews.isnull().sum().sum()}')
print(f'Duplicatas: {df_reviews.duplicated().sum()}')
print(f'review_id únicos: {df_reviews["review_id"].nunique()}')
print()
print(df_reviews.dtypes)
print()
print('✅ reviews - pronto para exportação Silver.')

=== VALIDAÇÃO FINAL — reviews ===
Shape: (1106, 6)
Nulos totais: 0
Duplicatas: 0
review_id únicos: 1106

review_id               int64
product_id              int64
customer_id             int64
rating                  int64
review_text               str
review_date    datetime64[us]
dtype: object

✅ reviews — pronto para exportação Silver.


---
## 9. `suppliers.csv` - Diagnóstico e Limpeza

**Tabela:** Fornecedores  
**Colunas esperadas:** `supplier_id`, `supplier_name`, `contact_name`, `address`, `phone_number`, `email`  
**Registros esperados:** 100

### 9.1 Carregamento

In [57]:
# Carregamento da tabela suppliers
df_suppliers = pd.read_csv(PATH_BRONZE / 'suppliers.csv')
print(f'Shape: {df_suppliers.shape}')
df_suppliers.head()

Shape: (100, 6)


,supplier_id,supplier_name,contact_name,address,phone_number,email
0,501,Dynamic Systems Group,Donald Benjamin,"141 Shore Ln, Island City, MA",(555) 484-6922,dbenjamin@supplier.com
1,502,Dynamic Systems Group,Nicholas Dennis,"161 Harbor Ln, Bay Point, FM",(555) 397-6986,ndennis@supplier.com
2,503,Mega Suppliers,Catherine Moore,"151 Pearl St, Seaside, SD",(555) 240-4096,cmoore@supplier.com
3,504,Tech Supplies Inc.,Linda Virginia,"505 Walnut St, Coast City, OR",(555) 624-2518,lvirginia@supplier.com
4,505,Ultimate Services,Mark Samuel,"585 Lighthouse Rd, Sea Haven, LA",(555) 295-9426,msamuel@supplier.com


### 9.2 Diagnóstico de Qualidade

In [58]:
# --- Tipos de dados ---
print('=== TIPOS DE DADOS ===')
print(df_suppliers.dtypes)
print()

=== TIPOS DE DADOS ===
supplier_id      int64
supplier_name      str
contact_name       str
address            str
phone_number       str
email              str
dtype: object



In [59]:
# --- Valores nulos ---
print('=== VALORES NULOS ===')
print(df_suppliers.isnull().sum())
print()

=== VALORES NULOS ===
supplier_id      0
supplier_name    0
contact_name     0
address          0
phone_number     0
email            0
dtype: int64



In [60]:
# --- Duplicatas (linhas completas) ---
n_duplicatas = df_suppliers.duplicated().sum()
print(f'=== DUPLICATAS (full row) === {n_duplicatas}')
print()

=== DUPLICATAS (full row) === 0



In [61]:
# --- Cardinalidade ---
print('=== CARDINALIDADE ===')
for col in df_suppliers.columns:
    print(f'  {col}: {df_suppliers[col].nunique()} únicos')
print()

=== CARDINALIDADE ===
  supplier_id: 100 únicos
  supplier_name: 25 únicos
  contact_name: 99 únicos
  address: 46 únicos
  phone_number: 64 únicos
  email: 92 únicos



In [62]:
# --- Análise: supplier_name (baixa cardinalidade) ---
print('=== DISTRIBUIÇÃO supplier_name ===')
print(df_suppliers['supplier_name'].value_counts())
print(f'\nTotal nomes únicos: {df_suppliers["supplier_name"].nunique()} para {len(df_suppliers)} registros')

=== DISTRIBUIÇÃO supplier_name ===
supplier_name
Next Level Systems           8
Ultimate Services            6
Modern Tech Enterprises      6
Mega Suppliers               5
Unified Trading Co.          5
                            ..
Alpha Industries Ltd.        3
Professional Supply Chain    3
Reliable Resources Inc.      3
Quantum Enterprises          2
Smart Solutions Ltd.         1
Name: count, Length: 25, dtype: int64

Total nomes únicos: 25 para 100 registros


In [63]:
# --- Análise: phone_number duplicados e formato ---
print('=== PHONE_NUMBER ===')
print(f'Únicos: {df_suppliers["phone_number"].nunique()} de {len(df_suppliers)}')
print(f'Duplicados: {len(df_suppliers) - df_suppliers["phone_number"].nunique()}')
print()
print('Exemplos de formato:')
print(df_suppliers['phone_number'].head(5).tolist())
print()
print('⚠️ Formato suppliers: (555) XXX-XXXX')
print('⚠️ Formato customers: 555-XXX-XXXX')
print('→ Inconsistência de formato entre tabelas (documentar).')

=== PHONE_NUMBER ===
Únicos: 64 de 100
Duplicados: 36

Exemplos de formato:
['(555) 484-6922', '(555) 397-6986', '(555) 240-4096', '(555) 624-2518', '(555) 295-9426']

⚠️ Formato suppliers: (555) XXX-XXXX
⚠️ Formato customers: 555-XXX-XXXX
→ Inconsistência de formato entre tabelas (documentar).


In [64]:
# --- Análise: email duplicados ---
print('=== EMAIL ===')
print(f'Únicos: {df_suppliers["email"].nunique()} de {len(df_suppliers)}')
print(f'Duplicados: {len(df_suppliers) - df_suppliers["email"].nunique()}')

=== EMAIL ===
Únicos: 92 de 100
Duplicados: 8


### 9.3 Justificativas e Decisões de Limpeza

| # | Observação | Decisão | Justificativa |
|---|---|---|---|
| 1 | **Nenhum valor nulo** encontrado | ✅ Nenhuma ação | Dataset completo |
| 2 | **Nenhuma duplicata** de linha completa | ✅ Nenhuma ação | IDs únicos |
| 3 | **Baixa cardinalidade em `supplier_name`** - apenas 25 nomes únicos para 100 fornecedores | 📝 Documentar como limitação | Dataset sintético - empresas com múltiplos registros/contatos |
| 4 | **Baixa cardinalidade em `address`** (46), `phone_number` (64), `email` (92) | 📝 Documentar como limitação | Característica do dataset sintético |
| 5 | **Formato de `phone_number`**: `(555) XXX-XXXX` - diferente de `customers.csv` (`555-XXX-XXXX`) | 📝 Documentar inconsistência | São tabelas distintas com contextos diferentes. Manter formatos originais, mas registrar a observação para eventual padronização futura. |
| 6 | **Tipos de dados** adequados e **colunas em `snake_case`** | ✅ Nenhuma ação | Corretos |

> **Conclusão:** O dataset `suppliers.csv` está **limpo**. Nenhuma transformação estrutural necessária. As limitações de cardinalidade e a inconsistência de formato de telefone entre tabelas são documentadas.

### 9.4 Limpeza e Validação

In [ ]:
# suppliers.csv não requer transformações - dataset já está limpo.
print('=== VALIDAÇÃO FINAL - suppliers ===')
print(f'Shape: {df_suppliers.shape}')
print(f'Nulos totais: {df_suppliers.isnull().sum().sum()}')
print(f'Duplicatas: {df_suppliers.duplicated().sum()}')
print(f'supplier_id únicos: {df_suppliers["supplier_id"].nunique()}')
print()
print(df_suppliers.dtypes)
print()
print('✅ suppliers - pronto para exportação Silver.')

=== VALIDAÇÃO FINAL — suppliers ===
Shape: (100, 6)
Nulos totais: 0
Duplicatas: 0
supplier_id únicos: 100

supplier_id      int64
supplier_name      str
contact_name       str
address            str
phone_number       str
email              str
dtype: object

✅ suppliers — pronto para exportação Silver.


---
## 10. `shipments.csv` - Diagnóstico e Limpeza

**Tabela:** Envios de pedidos  
**Colunas esperadas:** `shipment_id`, `order_id`, `shipment_date`, `carrier`, `tracking_number`, `delivery_date`, `shipment_status`  
**Registros esperados:** 15.000

### 10.1 Carregamento

In [66]:
# Carregamento da tabela shipments
df_shipments = pd.read_csv(PATH_BRONZE / 'shipments.csv')
print(f'Shape: {df_shipments.shape}')
df_shipments.head()

Shape: (15000, 7)


,shipment_id,order_id,shipment_date,carrier,tracking_number,delivery_date,shipment_status
0,1,1,2024-10-13,UPS,TRK344284,2024-10-14,Pending
1,2,2,2024-08-27,DHL,TRK718398,2024-08-28,Delivered
2,3,3,2024-05-23,FedEx,TRK161368,2024-05-31,Pending
3,4,4,2024-06-06,UPS,TRK890181,2024-06-10,Pending
4,5,5,2024-11-03,DHL,TRK681341,2024-11-08,Delivered


### 10.2 Diagnóstico de Qualidade

In [67]:
# --- Tipos de dados ---
print('=== TIPOS DE DADOS ===')
print(df_shipments.dtypes)
print()

=== TIPOS DE DADOS ===
shipment_id        int64
order_id           int64
shipment_date        str
carrier              str
tracking_number      str
delivery_date        str
shipment_status      str
dtype: object



In [68]:
# --- Valores nulos ---
print('=== VALORES NULOS ===')
print(df_shipments.isnull().sum())
print()

=== VALORES NULOS ===
shipment_id        0
order_id           0
shipment_date      0
carrier            0
tracking_number    0
delivery_date      0
shipment_status    0
dtype: int64



In [69]:
# --- Duplicatas (linhas completas) ---
n_duplicatas = df_shipments.duplicated().sum()
print(f'=== DUPLICATAS (full row) === {n_duplicatas}')
print()

=== DUPLICATAS (full row) === 0



In [70]:
# --- Cardinalidade ---
print('=== CARDINALIDADE ===')
for col in df_shipments.columns:
    print(f'  {col}: {df_shipments[col].nunique()} únicos')
print()

=== CARDINALIDADE ===
  shipment_id: 15000 únicos
  order_id: 15000 únicos
  shipment_date: 369 únicos
  carrier: 3 únicos
  tracking_number: 14904 únicos
  delivery_date: 373 únicos
  shipment_status: 4 únicos



In [71]:
# --- Análise: shipment_status ---
print('=== DISTRIBUIÇÃO shipment_status ===')
print(df_shipments['shipment_status'].value_counts())
print()

=== DISTRIBUIÇÃO shipment_status ===
shipment_status
Delivered    5374
Shipped      5337
Pending      3559
Cancelled     730
Name: count, dtype: int64



In [72]:
# --- Análise: carrier ---
print('=== DISTRIBUIÇÃO carrier ===')
print(df_shipments['carrier'].value_counts())
print()

=== DISTRIBUIÇÃO carrier ===
carrier
UPS      5000
DHL      5000
FedEx    5000
Name: count, dtype: int64



In [73]:
# --- Análise: datas como object ---
print('=== ANÁLISE: shipment_date e delivery_date ===')
print(f'shipment_date dtype: {df_shipments["shipment_date"].dtype}')
print(f'delivery_date dtype: {df_shipments["delivery_date"].dtype}')
print()
print(f'shipment_date range: {df_shipments["shipment_date"].min()} a {df_shipments["shipment_date"].max()}')
print(f'delivery_date range: {df_shipments["delivery_date"].min()} a {df_shipments["delivery_date"].max()}')

=== ANÁLISE: shipment_date e delivery_date ===
shipment_date dtype: str
delivery_date dtype: str

shipment_date range: 2023-11-06 a 2024-11-08
delivery_date range: 2023-11-09 a 2024-11-15


In [74]:
# --- Análise: delivery_date em status Pending/Cancelled ---
print('=== INCONSISTÊNCIA: delivery_date em Pending/Cancelled ===')
for status in ['Pending', 'Cancelled']:
    subset = df_shipments[df_shipments['shipment_status'] == status]
    has_delivery = subset['delivery_date'].notna().sum()
    print(f'  {status}: {len(subset)} registros, {has_delivery} com delivery_date preenchida')
print()
print('⚠️ Envios Pending/Cancelled NÃO deveriam ter data de entrega.')
print('   → Ação: substituir delivery_date por NaT nesses casos.')

=== INCONSISTÊNCIA: delivery_date em Pending/Cancelled ===
  Pending: 3559 registros, 3559 com delivery_date preenchida
  Cancelled: 730 registros, 730 com delivery_date preenchida

⚠️ Envios Pending/Cancelled NÃO deveriam ter data de entrega.
   → Ação: substituir delivery_date por NaT nesses casos.


In [75]:
# --- Análise: tracking_number duplicados ---
print('=== TRACKING_NUMBER DUPLICADOS ===')
trk_dupes = df_shipments[df_shipments.duplicated(subset=['tracking_number'], keep=False)]
n_trk_uniq = trk_dupes['tracking_number'].nunique()
print(f'Tracking numbers duplicados: {n_trk_uniq}')
print(f'Total de registros afetados: {len(trk_dupes)}')
print()
print('Exemplos (primeiros 5 tracking duplicados):')
print(trk_dupes.sort_values('tracking_number')[['shipment_id','tracking_number','shipment_status']].head(10))

=== TRACKING_NUMBER DUPLICADOS ===
Tracking numbers duplicados: 95
Total de registros afetados: 191

Exemplos (primeiros 5 tracking duplicados):
       shipment_id tracking_number shipment_status
28              29       TRK001293         Pending
12949        12950       TRK001293         Pending
12877        12878       TRK003202       Delivered
14857        14858       TRK003202         Shipped
2844          2845       TRK005714       Delivered
10661        10662       TRK005714         Shipped
971            971       TRK010446         Pending
5226          5227       TRK010446       Delivered
6828          6829       TRK036770         Pending
12669        12670       TRK036770       Delivered


### 10.3 Justificativas e Decisões de Limpeza

| # | Observação | Decisão | Justificativa |
|---|---|---|---|
| 1 | **Nenhum valor nulo** encontrado | ✅ Nenhuma ação | Dataset completo |
| 2 | **Nenhuma duplicata** de linha completa | ✅ Nenhuma ação | IDs únicos |
| 3 | **`shipment_date` e `delivery_date` como `object`** | 🔧 Converter para `datetime64` | Necessário para cálculos de tempo de entrega e análises temporais |
| 4 | **Inconsistência lógica:** envios `Pending` e `Cancelled` possuem `delivery_date` preenchida | 🔧 Substituir `delivery_date` por `NaT` | Um pedido pendente ou cancelado **não pode** ter data de entrega - o dado original é inconsistente. `NaT` representa corretamente a ausência lógica de entrega. |
| 5 | **96 `tracking_number` duplicados** (191 registros afetados) | 📝 Manter e documentar | Podem representar reenvios ou parcelas do mesmo pedido. Registrar anomalia para análise futura. |
| 6 | **`carrier`** com 3 valores (UPS, FedEx, DHL) | ✅ Nenhuma ação | Distribuição consistente |
| 7 | **Nenhuma entrega antes do envio** (`delivery_date ≥ shipment_date`) | ✅ Nenhuma ação | Consistência temporal OK |
| 8 | **Colunas já em `snake_case`** | ✅ Nenhuma ação | Padrão seguido |

> **Conclusão:** Duas transformações necessárias: (1) conversão de datas para `datetime64`, (2) limpeza de `delivery_date` para status `Pending`/`Cancelled` → `NaT`. Os tracking numbers duplicados são documentados.

### 10.4 Limpeza e Validação

In [76]:
# === Limpeza 1: Conversão de datas para datetime64 ===
print('=== ANTES DA CONVERSÃO ===')
print(f'shipment_date dtype: {df_shipments["shipment_date"].dtype}')
print(f'delivery_date dtype: {df_shipments["delivery_date"].dtype}')
print()

df_shipments['shipment_date'] = pd.to_datetime(df_shipments['shipment_date'])
df_shipments['delivery_date'] = pd.to_datetime(df_shipments['delivery_date'])

print('=== APÓS A CONVERSÃO ===')
print(f'shipment_date dtype: {df_shipments["shipment_date"].dtype}')
print(f'delivery_date dtype: {df_shipments["delivery_date"].dtype}')

=== ANTES DA CONVERSÃO ===
shipment_date dtype: str
delivery_date dtype: str

=== APÓS A CONVERSÃO ===
shipment_date dtype: datetime64[us]
delivery_date dtype: datetime64[us]


In [77]:
# === Limpeza 2: delivery_date → NaT para Pending e Cancelled ===
mask = df_shipments['shipment_status'].isin(['Pending', 'Cancelled'])
n_afetados = mask.sum()
print(f'Registros afetados (Pending + Cancelled): {n_afetados}')
print(f'delivery_date preenchidas antes: {df_shipments.loc[mask, "delivery_date"].notna().sum()}')
print()

df_shipments.loc[mask, 'delivery_date'] = pd.NaT

print(f'delivery_date preenchidas depois: {df_shipments.loc[mask, "delivery_date"].notna().sum()}')
print('✅ delivery_date limpa para status Pending e Cancelled.')

Registros afetados (Pending + Cancelled): 4289
delivery_date preenchidas antes: 4289

delivery_date preenchidas depois: 0
✅ delivery_date limpa para status Pending e Cancelled.


In [ ]:
# === Validação final ===
print('=== VALIDAÇÃO FINAL - shipments ===')
print(f'Shape: {df_shipments.shape}')
print(f'Nulos totais: {df_shipments.isnull().sum().sum()}')
print(f'  → delivery_date NaT (esperados = Pending + Cancelled): {df_shipments["delivery_date"].isna().sum()}')
print(f'Duplicatas: {df_shipments.duplicated().sum()}')
print(f'shipment_id únicos: {df_shipments["shipment_id"].nunique()}')
print()
print(df_shipments.dtypes)
print()
# Verificar que Delivered/Shipped mantêm delivery_date
for st in ['Delivered', 'Shipped']:
    sub = df_shipments[df_shipments['shipment_status'] == st]
    print(f'  {st}: {sub["delivery_date"].notna().sum()}/{len(sub)} com delivery_date')
for st in ['Pending', 'Cancelled']:
    sub = df_shipments[df_shipments['shipment_status'] == st]
    print(f'  {st}: {sub["delivery_date"].isna().sum()}/{len(sub)} com NaT (esperado)')
print()
print('✅ shipments - pronto para exportação Silver.')

=== VALIDAÇÃO FINAL — shipments ===
Shape: (15000, 7)
Nulos totais: 4289
  → delivery_date NaT (esperados = Pending + Cancelled): 4289
Duplicatas: 0
shipment_id únicos: 15000

shipment_id                 int64
order_id                    int64
shipment_date      datetime64[us]
carrier                       str
tracking_number               str
delivery_date      datetime64[us]
shipment_status               str
dtype: object

  Delivered: 5374/5374 com delivery_date
  Shipped: 5337/5337 com delivery_date
  Pending: 3559/3559 com NaT (esperado)
  Cancelled: 730/730 com NaT (esperado)

✅ shipments — pronto para exportação Silver.


---
## 11. Exportação para Camada Silver (Parquet)

Salvamento de **todas** as tabelas limpas no formato `.parquet` na pasta `datasets/dataset_silver/`.

In [79]:
# === Exportação de todas as tabelas ===
tabelas = {
    'customers': df_customers,
    'order_items': df_order_items,
    'orders': df_orders,
    'payment': df_payment,
    'products': df_products,
    'reviews': df_reviews,
    'suppliers': df_suppliers,
    'shipments': df_shipments,
}

caminhos = {}
for nome, df in tabelas.items():
    caminho = PATH_SILVER / f'{nome}.parquet'
    df.to_parquet(caminho, index=False)
    caminhos[nome] = caminho
    print(f'✅ {nome}.parquet salvo | Registros: {len(df)} | Colunas: {len(df.columns)}')

print()
print(f' Destino: {PATH_SILVER.resolve()}')

✅ customers.parquet salvo | Registros: 10000 | Colunas: 6
✅ order_items.parquet salvo | Registros: 20000 | Colunas: 5
✅ orders.parquet salvo | Registros: 15000 | Colunas: 4
✅ payment.parquet salvo | Registros: 15000 | Colunas: 5
✅ products.parquet salvo | Registros: 2000 | Colunas: 5
✅ reviews.parquet salvo | Registros: 1106 | Colunas: 6
✅ suppliers.parquet salvo | Registros: 100 | Colunas: 6
✅ shipments.parquet salvo | Registros: 15000 | Colunas: 7

 Destino: C:\Users\Nogueira\Documents\GitHub\Trabalho_ciencia_de_dados\datasets\dataset_silver


In [ ]:
# === Verificação pós-exportação ===
print('=== VERIFICAÇÃO PÓS-EXPORTAÇÃO ===')
print()

for nome, caminho in caminhos.items():
    df_check = pd.read_parquet(caminho)
    tamanho_kb = os.path.getsize(caminho) / 1024
    print(f'📄 {nome}.parquet')
    print(f'   Shape: {df_check.shape}')
    print(f'   Tamanho: {tamanho_kb:.1f} KB')
    print(f'   Dtypes: {dict(df_check.dtypes)}')
    print()

print('='*60)
print('✅ EXPORTAÇÃO COMPLETA - Pipeline Bronze → Silver finalizado!')
print(f'   Total de tabelas: {len(caminhos)}')
print('   Formato: Parquet')
print(f'   Destino: {PATH_SILVER.resolve()}')

=== VERIFICAÇÃO PÓS-EXPORTAÇÃO ===

📄 customers.parquet
   Shape: (10000, 6)
   Tamanho: 249.1 KB
   Dtypes: {'customer_id': dtype('int64'), 'first_name': <StringDtype(na_value=nan)>, 'last_name': <StringDtype(na_value=nan)>, 'address': <StringDtype(na_value=nan)>, 'email': <StringDtype(na_value=nan)>, 'phone_number': <StringDtype(na_value=nan)>}

📄 order_items.parquet
   Shape: (20000, 5)
   Tamanho: 358.4 KB
   Dtypes: {'order_item_id': dtype('int64'), 'order_id': dtype('int64'), 'product_id': dtype('int64'), 'quantity': dtype('int64'), 'price_at_purchase': dtype('float64')}

📄 orders.parquet
   Shape: (15000, 4)
   Tamanho: 271.2 KB
   Dtypes: {'order_id': dtype('int64'), 'order_date': dtype('<M8[us]'), 'customer_id': dtype('int64'), 'total_price': dtype('float64')}

📄 payment.parquet
   Shape: (15000, 5)
   Tamanho: 232.5 KB
   Dtypes: {'payment_id': dtype('int64'), 'order_id': dtype('int64'), 'payment_method': <StringDtype(na_value=nan)>, 'amount': dtype('float64'), 'transaction_s

## 12. Comparação Bronze × Silver

A tabela abaixo resume as diferenças entre as duas camadas para cada tabela do dataset:

| Tabela | Bronze (CSV) | Silver (Parquet) | Transformações aplicadas |
|---|---|---|---|
| `customers` | 10.000 linhas, tipos object/int64 | 10.000 linhas, mesmos tipos | Nenhuma - dataset já estava limpo |
| `order_items` | 20.000 linhas | 20.000 linhas | Nenhuma - 2 registros com preço zero mantidos e documentados |
| `orders` | 15.000 linhas, `order_date` como object | 15.000 linhas, `order_date` como datetime64 | Conversão de tipo de data |
| `payment` | linhas variadas, `payment_date` como object | mesma contagem, `payment_date` como datetime64 | Conversão de tipo de data |
| `products` | linhas variadas | mesmas linhas | Nenhuma - dataset já estava limpo |
| `reviews` | 1.106 linhas, `review_date` como object | 1.106 linhas, `review_date` como datetime64 | Conversão de tipo de data |
| `suppliers` | 100 linhas | 100 linhas | Nenhuma - dataset já estava limpo |
| `shipments` | 15.000 linhas, datas como object, `delivery_date` inconsistente | 15.000 linhas, datas como datetime64, `delivery_date` = NaT para Pending/Cancelled | Conversão de tipos de data + limpeza de inconsistência lógica |

**Mudanças estruturais:** nenhuma tabela teve linhas removidas. As únicas alterações foram correções de tipo de dado e uma limpeza de inconsistência lógica em `shipments`. A contagem de registros é preservada integralmente entre as camadas.

**Formato:** a camada Bronze usa `.csv` (texto); a camada Silver usa `.parquet` (binário colunar), que oferece leitura mais eficiente, preservação nativa de tipos de dados (incluindo `datetime64`) e menor tamanho em disco.
